In [10]:
import numpy as np

In [11]:
def entropy(probs):
    probs = probs[probs>0]
    return -sum(probs * np.log2(probs))


In [12]:
# root entropy
IGroot = entropy(np.array([4/9, 5/9]))

In [13]:
print("a1 T", entropy(np.array([3/4, 1/4])))
print("a1 F", entropy(np.array([1/5, 4/5])))
IGa1 = 0.9910760598382222 -4/9*0.8112781244591328-5/9*0.7219280948873623
print(IGa1)

a1 T 0.8112781244591328
a1 F 0.7219280948873623
0.2294368406967397


In [14]:
a2Tentropy = entropy(np.array([2/5, 3/5]))
a2Fentropy = entropy(np.array([2/4, 2/4]))
print("a2 T", a2Tentropy)
print("a2 F", a2Fentropy)
IGa2 = 0.9910760598382222 - 5/9*a2Tentropy - 4/9 * a2Fentropy
print(IGa2)

a2 T 0.9709505944546686
a2 F 1.0
0.007214618474517431


In [15]:
a31entropy = entropy(np.array([1/1, 0]))
a33entropy = entropy(np.array([0, 1/1]))
a34entropy = entropy(np.array([1/1, 0]))
a35entropy = entropy(np.array([0, 2/2]))
a36entropy = entropy(np.array([1/1, 0]))
a37entropy = entropy(np.array([1/2, 1/2]))
a38entropy = entropy(np.array([0, 1/1]))
print("a3 1", a31entropy)
print("a3 3", a33entropy)
print("a3 4", a34entropy)
print("a3 5", a35entropy)
print("a3 6", a36entropy)
print("a3 7", a37entropy)
print("a3 8", a38entropy)
entropies = np.array([a31entropy, a33entropy, a34entropy, a35entropy, a36entropy, a37entropy, a38entropy])
weights = np.array([1/9, 1/9, 1/9, 2/9, 1/9, 2/9, 1/9])
IGa3 = 0.9910760598382222 - sum(entropies*weights)
print("IGa3:", IGa3)

a3 1 -0.0
a3 3 -0.0
a3 4 -0.0
a3 5 -0.0
a3 6 -0.0
a3 7 1.0
a3 8 -0.0
IGa3: 0.768853837616


In [18]:
print(entropy(np.array([3/8, 5/8])))

0.954434002924965


In [21]:
def IG(currnode_ent, entropies, weights):
    return currnode_ent - sum(weights*entropies)

In [25]:
IG(0.9910760598382222, 
   np.array([0, 0.954434002924965]), 
   np.array([0, 8/9]))

np.float64(0.14269027946047552)

In [29]:
import pandas as pd
df = pd.DataFrame({
    'a1':[True, True, True, False, False, False, False, True, False],
    'a2':[True, True, False, False, True, True, False, False, True],
    'a3':[1.0, 6.0, 5.0, 4.0, 7.0, 3.0, 8.0, 7.0, 5.0],
    'Class':[True, True, False, True, False, False, False, True, False]
})
def entropy(subset, class_col='Class'):
    counts = subset[class_col].value_counts()
    probs = counts / len(subset)
    return -np.sum(probs * np.log2(probs))

In [95]:
def entropy_verbose(subset, class_col='Class'):
    counts = subset[class_col].value_counts()
    total = len(subset)
    probs = counts / total
    terms = "-".join([f"({c}/{total}*log({c}/{total}))" for c in counts])
    h = -np.sum(probs * np.log2(probs))
    print(f"    H = -({terms})={h}")
    return h
def information_gain_verbose(df, attribute, threshold=None, class_col='Class'):
    n = len(df)
    
    
    # Step 1 - full dataset entropy
    counts = df[class_col].value_counts()
    print(f"=== IG for {attribute}{f' (threshold={threshold})' if threshold else ''} ===")
    print(f"Full dataset: {dict(counts)}, total={n}")
    print(f"H(S):")
    H_S = entropy_verbose(df)
    
    # Step 2 - split
    if threshold is not None:
        left  = df[df[attribute] < threshold]
        right = df[df[attribute] >= threshold]
        subsets = [(f"{attribute} < {threshold}", left), (f"{attribute} >= {threshold}", right)]
    else:
        subsets = [(f"{attribute}={v}", df[df[attribute] == v]) for v in sorted(df[attribute].unique())]
    
    # Step 3 - weighted entropy
    print(f"\nSubsets:")
    weighted_entropy = 0
    subset_info = []
    for name, s in subsets:
        if len(s) == 0:
            continue
        counts_s = dict(s[class_col].value_counts())
        print(f"  {name}: {counts_s}, total={len(s)}")
        h = entropy_verbose(s)
        w = len(s) / n
        weighted_entropy += w * h
        subset_info.append((name, len(s), h))
    
    # Print weighted entropy equation
    weighted_eq = "+".join([f"({size}/{n}*{h})" for name, size, h in subset_info])
    print(f"\nWeighted entropy = {weighted_eq}={weighted_entropy}")
    
    # Step 4 - IG
    ig = H_S - weighted_entropy
    print(f"IG={H_S}-{weighted_entropy}={ig}")
    print()
    return ig

In [90]:
def calculate_midpoints(l):
    l = sorted(l)
    k = []
    for i in range(1, len(l)):
        k.append((l[i-1]+l[i])/2)
    print(k)
    return k


In [86]:
information_gain_verbose(df, 'a3', threshold=5.5)

=== IG for a3 (threshold=5.5) ===
Full dataset: {False: np.int64(5), True: np.int64(4)}, total=9
H(S):
    H = -((5/9 * log2(5/9)) - (4/9 * log2(4/9)))
      = 0.9910760598382222

Subsets:
  a3 < 5.5: {False: np.int64(3), True: np.int64(2)}, total=5
    H = -((3/5 * log2(3/5)) - (2/5 * log2(2/5)))
      = 0.9709505944546686
  a3 >= 5.5: {True: np.int64(2), False: np.int64(2)}, total=4
    H = -((2/4 * log2(2/4)) - (2/4 * log2(2/4)))
      = 1.0

Weighted entropy = (5/9 * 0.9709505944546686) + (4/9 * 1.0)
                 = 0.9838614413637048
IG = 0.9910760598382222 - 0.9838614413637048 = 0.007214618474517431



np.float64(0.007214618474517431)

In [64]:
a1_left = df[df['a1']==True]

In [77]:
calculate_midpoints(list(a1_left['a3']))

[1.0, 5.0, 6.0, 7.0]


[3.0, 5.5, 6.5]

In [94]:
for mid in calculate_midpoints(list(a1_left['a3'])):
    information_gain_verbose(a1_left, 'a3', threshold=mid)


[3.0, 5.5, 6.5]
=== IG for a3 (threshold=3.0) ===
Full dataset: {True: np.int64(3), False: np.int64(1)}, total=4
H(S):
    H = -((3/4*log_2(3/4))-(1/4*log_2(1/4)))=0.8112781244591328

Subsets:
  a3 < 3.0: {True: np.int64(1)}, total=1
    H = -((1/1*log_2(1/1)))=-0.0
  a3 >= 3.0: {True: np.int64(2), False: np.int64(1)}, total=3
    H = -((2/3*log_2(2/3))-(1/3*log_2(1/3)))=0.9182958340544896

Weighted entropy = (1/4*-0.0)+(3/4*0.9182958340544896)=0.6887218755408672
IG=0.8112781244591328-0.6887218755408672=0.12255624891826566

=== IG for a3 (threshold=5.5) ===
Full dataset: {True: np.int64(3), False: np.int64(1)}, total=4
H(S):
    H = -((3/4*log_2(3/4))-(1/4*log_2(1/4)))=0.8112781244591328

Subsets:
  a3 < 5.5: {True: np.int64(1), False: np.int64(1)}, total=2
    H = -((1/2*log_2(1/2))-(1/2*log_2(1/2)))=1.0
  a3 >= 5.5: {True: np.int64(2)}, total=2
    H = -((2/2*log_2(2/2)))=-0.0

Weighted entropy = (2/4*1.0)+(2/4*-0.0)=0.5
IG=0.8112781244591328-0.5=0.31127812445913283

=== IG for a3 (t

In [84]:
-(((1/3 * np.log2(1/3)) + (2/3 * np.log2(2/3))))

np.float64(0.9182958340544896)